### 과제 내용
### 1. 데이터를 pandas DataFrame으로 만들고, groupby 기능을 사용해보세요.
### 2. 데이터를 DataFrame으로 만들고, 특정 조건을 필터링한 DataFrame을 만드세요.

# Top 500 LLM Models on HuggingFace Dataset

## 데이터셋 개요
HuggingFace에 등록된 상위 500개 LLM 모델의 다운로드 수, 좋아요, 라이브러리, 태그 등의 정보가 담긴 dataset

---

##  컬럼 설명

| 컬럼 | 설명 |
|---|---|
| `model_id` | 모델 식별자 (조직명/모델명 형식) |
| `downloads` | 총 다운로드 수 |
| `likes` | 좋아요 수 |
| `library` | 사용 라이브러리 (transformers, diffusers 등) |
| `tags` | 모델 태그 (라이선스, 태스크 등 메타정보) |
| `created_at` | 모델 최초 등록일 |
| `last_modified` | 마지막 수정일 |
| `private` | 비공개 여부 (True / False) |

In [16]:
import pandas as pd

df = pd.read_csv('datasets/Top_500_LLM_Models_HuggingFace.csv')

df['organization'] = df['model_id'].apply(lambda x: x.split('/')[0] if '/' in x else x)

# 연도 컬럼 추출
df['year'] = pd.to_datetime(df['created_at'], errors='coerce').dt.year

print('데이터 크기:', df.shape)
df.head()

데이터 크기: (500, 10)


,model_id,downloads,likes,library,tags,created_at,last_modified,private,organization,year
0,Qwen/Qwen3-0.6B,19369646,1221,transformers,"transformers, safetensors, qwen3, text-generat...",2025-04-27T03:40:08.000Z,NaN,False,Qwen,2025
1,openai-community/gpt2,16037172,3226,transformers,"transformers, pytorch, tf, jax, tflite, rust, ...",2022-03-02T23:29:04.000Z,NaN,False,openai-community,2022
2,Qwen/Qwen2.5-7B-Instruct,13784608,1253,transformers,"transformers, safetensors, qwen2, text-generat...",2024-09-16T11:55:40.000Z,NaN,False,Qwen,2024
3,deepseek-ai/DeepSeek-V3.2,11349614,1427,transformers,"transformers, safetensors, deepseek_v32, text-...",2025-12-01T02:34:49.000Z,NaN,False,deepseek-ai,2025
4,Qwen/Qwen3-4B-Instruct-2507,10691206,829,transformers,"transformers, safetensors, qwen3, text-generat...",2025-08-05T10:58:03.000Z,NaN,False,Qwen,2025


In [17]:
# 파라미터 단위별 평균 다운로드 수

import re

# model_id에서 파라미터 수 추출 (예: 7B, 70B, 0.6B)
def extract_params(model_id):
    match = re.search(r'(\d+\.?\d*)[Bb]', model_id)
    return float(match.group(1)) if match else None

df['params_B'] = df['model_id'].apply(extract_params)

# 파라미터 수 구간화
df['param_group'] = pd.cut(
    df['params_B'],
    bins=[0, 1, 7, 13, 70, float('inf')],
    labels=['~1B', '1~7B', '7~13B', '13~70B', '70B+']
)

# 구간별 평균 좋아요 수
df.groupby('param_group', observed=True).agg(
    모델수=('model_id', 'count'),
    평균좋아요=('likes', 'mean'),
    평균다운로드=('downloads', 'mean')
).round(0)

,모델수,평균좋아요,평균다운로드
param_group,,,
~1B,65,490.0,3599076.0
1~7B,140,737.0,3075480.0
7~13B,45,2333.0,3458366.0
13~70B,95,751.0,2174217.0
70B+,10,2522.0,2539065.0


In [18]:
# ② 연도별 모델 출시 수와 총 다운로드 수

df.groupby('year').agg(
    출시모델수=('model_id', 'count'),
    총다운로드=('downloads', 'sum'),
    평균좋아요=('likes', 'mean')
).round(0).sort_values('year')

,출시모델수,총다운로드,평균좋아요
year,,,
2022,35,159495725,689.0
2023,35,69120865,1595.0
2024,230,610938260,847.0
2025,150,553016030,1326.0
2026,50,64578010,392.0


In [19]:
# ③ 모델 만든 회사별 총 다운로드 수 Top 10

df.groupby('organization').agg(
    모델수=('model_id', 'count'),
    총다운로드=('downloads', 'sum'),
    총좋아요=('likes', 'sum')
).sort_values('총다운로드', ascending=False).head(10)

,모델수,총다운로드,총좋아요
organization,,,
Qwen,185,680355140,89595
meta-llama,45,143856235,139710
deepseek-ai,40,112832085,121965
openai-community,15,95917255,18870
openai,10,55640690,46670
nvidia,35,43931995,9715
facebook,5,39560160,1245
trl-internal-testing,5,36272790,30
EleutherAI,10,24065580,345


In [20]:
# ④ 다운로드 구간별 평균 좋아요 수

df['download_group'] = pd.cut(
    df['downloads'],
    bins=[0, 1_000_000, 3_000_000, 7_000_000, df['downloads'].max()],
    labels=['~100만', '100~300만', '300~700만', '700만+']
)

df.groupby('download_group', observed=True).agg(
    모델수=('model_id', 'count'),
    평균좋아요=('likes', 'mean'),
    평균다운로드=('downloads', 'mean')
).round(0)

,모델수,평균좋아요,평균다운로드
download_group,,,
~100만,105,187.0,861238.0
100~300만,270,805.0,1598060.0
300~700만,70,2504.0,4465782.0
700만+,55,1472.0,11320688.0


In [21]:
df['year'] = pd.to_datetime(df['created_at']).dt.year
# 연도 × 라이브러리별 출시 모델 수
df.groupby(['year', 'library']).size().unstack(fill_value=0).sort_index()


library,Model Optimizer,transformers
year,,
2022,0,35
2023,0,35
2024,0,210
2025,5,145
2026,10,25
